Część 1: Podłączenie Sparka do Kafki

In [3]:
import os
import pyspark

# Autodetekcja wersji Sparka → właściwy connector
spark_version = pyspark.__version__
print(f"Wykryta wersja PySpark: {spark_version}")

if spark_version.startswith("4"):
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
else:
    # Spark 3.x
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"

print(f"Użyty connector:        {KAFKA_PACKAGE}")

# Musi być ustawione PRZED SparkSession.builder
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages {KAFKA_PACKAGE} pyspark-shell'

Wykryta wersja PySpark: 4.0.0.dev2
Użyty connector:        org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


Część 2: Co naprawdę przychodzi z Kafki?

In [6]:
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)

kafka_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [5]:
%%file kafka_raw.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
# spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2

spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)

q = (
    kafka_raw.writeStream
    .format("console") 
    .outputMode("append") 
    .option("truncate", False)
    .start()
)

q.awaitTermination()

Overwriting kafka_raw.py


Część 3: Krok po kroku — bajty → tabela

In [4]:
%%file kafka_text.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
# spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2

spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)

df = kafka_raw.select(
    col("value").cast("string")
)

q = (
    df.writeStream
    .format("console") 
    .outputMode("append") 
    .option("truncate", False)
    .start()
)

q.awaitTermination()

Overwriting kafka_text.py


In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json, col

spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)


tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])

# Krok 2: string → struct (jedna kolumna 'tx' zawierająca wszystkie pola)
step2 = kafka_raw.select(
    from_json(col("value").cast("string"), tx_schema).alias("tx")
)

query = (step2.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+---+
|tx |
+---+
+---+

Batch ID: 1
+---------------------------------------------------------------------+
|tx                                                                   |
+---------------------------------------------------------------------+
|{TX3313, u08, 3230.88, Warszawa, żywność, 2026-05-01T08:25:20.570671}|
+---------------------------------------------------------------------+

Batch ID: 2
+---------------------------------------------------------------------+
|tx                                                                   |
+---------------------------------------------------------------------+
|{TX4892, u10, 3794.49, Warszawa, żywność, 2026-05-01T08:25:21.571115}|
|{TX6959, u08, 3006.45, Warszawa, książki, 2026-05-01T08:25:22.571627}|
+---------------------------------------------------------------------+

Batch ID: 3
+----------------------------------------------------------------+
|tx                                                              |

In [10]:
from pyspark.sql.functions import to_timestamp

# Krok 3: struct → płaskie kolumny + konwersja timestamp
df = (
    kafka_raw
    .select(from_json(col("value").cast("string"), tx_schema).alias("tx"))
    .select("tx.*")                                         # struct → kolumny
    .withColumn("timestamp", to_timestamp("timestamp"))
)

print("Finalny schemat:")
df.printSchema()

Finalny schemat:
root
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- store: string (nullable = true)
 |-- category: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



Część 4: Okna tumbling + watermark

In [11]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round

windowed = (
    df
    .withWatermark("timestamp", "3 seconds")
    .groupBy(window("timestamp", "60 seconds"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
)


In [ ]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round

windowed_category = (
    df
    .withWatermark("timestamp", "3 seconds")
    .groupBy(window("timestamp", "20 seconds"), "category")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
)

query = (
    windowed_category
    .writeStream
    .format("console")
    .outputMode("complete")
    .option("truncate", False)
    .start()
)

#query.awaitTermination()

Część 5: Zapis alertów do Kafki

In [ ]:
from pyspark.sql.functions import to_json, struct, lit

alerts = (
    df
    .filter(col("amount") > 3000)
    .select(
        to_json(
            struct(
                "tx_id", "user_id", "amount", "store", "category",
                col("timestamp").cast("string"),
                lit("HIGH").alias("alert_level"),
            )
        ).alias("value")    # Kafka wymaga kolumny 'value'
    )
)

alert_query = (
    alerts.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic", "alerts")
    .outputMode("append")
    .start()
)
print("Strumień alertów uruchomiony. Zatrzymaj ręcznie: alert_query.stop()")

In [10]:
df.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- store: string (nullable = true)
 |-- category: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [11]:
from pyspark.sql.functions import to_json, struct, lit, col

alerts = (
    df
    .filter(col("amount") > 100)
    .select(
        to_json(
            struct(
                "tx_id",
                "user_id",
                "amount",
                "store",
                "category",
                col("timestamp").cast("string").alias("timestamp"),
                lit("HIGH").alias("alert_level")
            )
        ).alias("value")
    )
)

alert_query = (
    alerts.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic", "alerts")
    .option("checkpointLocation", "/tmp/checkpoints/alerts_test")
    .outputMode("append")
    .start()
)

print("Alert stream active:", alert_query.isActive)

Alert stream active: True


In [ ]:
# 1. Dlaczego w trybie append pierwsze wyniki pojawiają się z opóźnieniem?
# W trybie append Spark wypisuje wynik dopiero wtedy, gdy uzna okno czasowe za zamknięte.
# Przy agregacjach okienkowych musi jeszcze poczekać na ewentualne spóźnione dane,
# zgodnie z ustawionym watermarkiem. Dlatego pierwsze wyniki nie pojawiają się od razu,
# tylko dopiero po zakończeniu okna i minięciu czasu opóźnienia.

# 2. Co się stanie, jeśli ustawisz watermark na "0 seconds"?
# Spark praktycznie nie będzie czekał na spóźnione dane. Okna będą zamykane szybciej,
# ale zdarzenia, które dotrą po czasie, mogą zostać odrzucone i nie trafią do agregacji.
# To daje szybsze wyniki, ale zwiększa ryzyko utraty części danych.

# 3. Jaka jest różnica między window() w Lab 2 (batch) a window() tutaj (streaming)?
# W batchu dane są już kompletne, więc window() tylko grupuje istniejący zbiór danych
# według przedziałów czasu. W streamingu dane napływają ciągle, więc Spark musi
# aktualizować agregacje w kolejnych batchach i decydować, kiedy okno można zamknąć.
# Dlatego w streamingu potrzebny jest watermark, który obsługuje spóźnione zdarzenia.

# 4. Dlaczego do zapisu do Kafki używamy to_json(), a nie po prostu select("value")?
# Kafka oczekuje kolumny value zapisanej jako tekst lub bajty. Nasze dane po obróbce
# są wieloma kolumnami, np. tx_id, amount, store i timestamp. Funkcja to_json()
# pakuje te kolumny z powrotem do jednego tekstowego JSON-a, który można wysłać
# jako kolumnę value do tematu Kafka.

Praca domowa

In [13]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round

windowed_hw = (
    df
    .withWatermark("timestamp", "10 seconds")
    .groupBy(window("timestamp", "2 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
)

query_hw = (
    windowed_hw
    .writeStream
    .format("console")
    .outputMode("complete")
    .option("truncate", False)
    .start()
)

print("Stream uruchomiony.")

Stream uruchomiony.


In [14]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round, current_timestamp, col, unix_timestamp

windowed_delay = (
    df
    .withWatermark("timestamp", "10 seconds")
    .groupBy(window("timestamp", "2 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "store",
        "liczba_tx",
        "suma_PLN",
        current_timestamp().alias("processing_time"),
        (
            unix_timestamp(current_timestamp()) - unix_timestamp(col("window.end"))
        ).alias("opoznienie_sekundy")
    )
)

In [15]:
query_delay = (
    windowed_delay
    .writeStream
    .format("console")
    .outputMode("complete")
    .option("truncate", False)
    .start()
)

In [16]:
query_delay.isActive

True

In [17]:
for q in spark.streams.active:
    print("Zatrzymuję:", q.id)
    q.stop()

Zatrzymuję: e02ca7ce-918b-403f-9f4a-10a4cb49f010
Zatrzymuję: 41a8146c-1058-42b6-beea-3f86c2dbc649
Zatrzymuję: 31ed9214-3fea-421b-97c5-4b8284a52c7c
Zatrzymuję: 1593d2d1-60c7-4568-93bf-4c0aea50d3f4
Zatrzymuję: e385315e-c288-441e-bfe1-6436320539e1


In [18]:
query_delay = (
    windowed_delay
    .writeStream
    .format("memory")
    .queryName("wyniki_delay")
    .outputMode("complete")
    .start()
)

print("Stream delay uruchomiony:", query_delay.isActive)

Stream delay uruchomiony: True


In [20]:
spark.sql("""
SELECT *
FROM wyniki_delay
ORDER BY window_start, store
""").show(truncate=False)

+-------------------+-------------------+--------+---------+--------+-----------------------+------------------+
|window_start       |window_end         |store   |liczba_tx|suma_PLN|processing_time        |opoznienie_sekundy|
+-------------------+-------------------+--------+---------+--------+-----------------------+------------------+
|2026-05-01 09:06:00|2026-05-01 09:08:00|Gdańsk  |9        |29482.57|2026-05-01 09:07:27.891|-33               |
|2026-05-01 09:06:00|2026-05-01 09:08:00|Kraków  |8        |20006.48|2026-05-01 09:07:27.891|-33               |
|2026-05-01 09:06:00|2026-05-01 09:08:00|Warszawa|11       |36114.89|2026-05-01 09:07:27.891|-33               |
|2026-05-01 09:06:00|2026-05-01 09:08:00|Wrocław |4        |8933.37 |2026-05-01 09:07:27.891|-33               |
+-------------------+-------------------+--------+---------+--------+-----------------------+------------------+



In [21]:
for q in spark.streams.active:
    print("Zatrzymuję:", q.id)
    q.stop()

Zatrzymuję: e3bdd0fa-4898-4b43-9276-8577b43a1729
